# PCA Series | Chapter 1: Geometry & Applications

> **Chapter Goal:** Understand PCA's geometric meaning — why it's a rotation that aligns axes with the data ellipse — and master its practical applications: visualization, noise reduction, image compression, preprocessing pipelines, and whitening. Know when PCA helps and when it actively hurts.

---

## 1. The Geometric View: Data Ellipse and PC Axes

### **The Data Ellipse**
For any $d$-dimensional dataset with covariance matrix $\Sigma$, the **data ellipse** (or ellipsoid in $d>2$ dims) captures the shape and orientation of the data cloud:
- The **axes of the ellipse** point in the eigenvector directions of $\Sigma$.
- The **half-lengths of the axes** are $\sqrt{\lambda_i}$ — the standard deviation along each principal direction.
- The $n\sigma$ contour (containing $\approx 95\%$ of Gaussian data for $n=2$) is an ellipse: $\{\vec{x}: (\vec{x}-\vec{\mu})^T \Sigma^{-1} (\vec{x}-\vec{\mu}) = n^2\}$.

### **PCA = Rotation to Align Axes with Ellipse**
Standard coordinates ($x_1, x_2, ..., x_d$) are not aligned with the data ellipse — the ellipse is tilted. PCA finds the rigid rotation $W$ (the matrix of eigenvectors) that aligns the coordinate axes with the ellipse axes:

$$\underbrace{\text{Original space}}_\text{tilted ellipse} \xrightarrow{Z = XW} \underbrace{\text{PC space}}_\text{axis-aligned ellipse}$$

After the rotation:
- The data ellipse is axis-aligned in PC space.
- The covariance matrix of $Z$ = $\Lambda$ (diagonal) — no off-diagonal terms.
- PC1 ($z$-axis 1) runs through the longest axis of the ellipse.
- PC2 ($z$-axis 2) runs through the second-longest axis, perpendicular to PC1.

### **The Loading Matrix $W$: How to Read It**
PC $k$ = $\vec{w}_k = [w_{1k}, w_{2k}, ..., w_{dk}]^T$ where $w_{jk}$ is the **loading** of original feature $j$ on PC $k$.

Interpretation: $w_{jk}$ is the correlation between the original feature $j$ and the $k$-th PC score.

- Large $|w_{jk}|$ → feature $j$ contributes strongly to PC $k$.
- Sign of $w_{jk}$ → direction of contribution (positive = high feature $j$ → high PC $k$ score).

**The biplot** combines PC scores (scatter) and loadings (arrows) in one plot for interpretation.

---

## 2. PCA for Visualization

The most common use of PCA: reduce high-dimensional data to $k=2$ or $k=3$ dimensions so it can be plotted.

### **Procedure**
1. Standardize data.
2. Apply PCA with $k=2$.
3. Plot the 2D scores, color-coded by label (if available).
4. Add loading arrows to interpret what the axes mean (biplot).

### **What Visualization Shows**
- **Clusters:** Groups visible in 2D PC space → classes are separable along high-variance directions.
- **Outliers:** Points far from the main cloud in PC space → unusually large projections.
- **Gradients:** Continuous variation along a PC → a latent continuous factor.

### **Critical Caveat**
PCA only shows you the **two directions of highest variance** — not necessarily the most discriminative. Two classes may be completely separable in a low-variance direction that PCA discards. If visualization shows overlap, the classes might still be separable in a PCA direction that wasn't plotted. For supervised visualization, use LDA instead.

---

## 3. PCA for Noise Reduction

### **The Signal vs Noise Model**
Real data often has the structure:
$$X_{\text{observed}} = X_{\text{signal}} + \epsilon_{\text{noise}}$$

Signal occupies a low-rank subspace (large eigenvalues). Noise is spread across all directions (small eigenvalues — it contributes roughly equally to all PCs).

### **The Denoising Procedure**
1. Project data onto top-$k$ PCs: $Z = X W_k$
2. Reconstruct: $\hat{X} = Z W_k^T + \vec{\mu}$

The reconstruction $\hat{X}$ is a **smoothed version** of $X$ — the high-frequency noise (in low-eigenvalue directions) has been removed.

**Why it works:** Signal eigenvectors have large eigenvalues because the signal creates strong correlations (variance). Noise eigenvectors have small eigenvalues because noise is incoherent (uncorrelated across data points). Keeping only the large eigenvalue directions filters out noise.

### **Choosing $k$ for Denoising**
Use the scree plot: signal eigenvectors appear above the "noise floor" — a flat plateau of roughly equal small eigenvalues. The elbow = where signal ends and noise begins.

For statistically principled $k$: use the **Marchenko-Pastur threshold** (Chapter 2).

---

## 4. PCA for Compression: Eigenfaces

**Eigenfaces** (Turk & Pentland, 1991) is one of the most famous applications of PCA — face recognition via PCA compression.

### **The Setup**
- Dataset: $N$ face images, each $h \times w$ pixels → vectorized to $\vec{x}_i \in \mathbb{R}^{hw}$.
- Data matrix: $X \in \mathbb{R}^{N \times hw}$ (e.g., 400 faces, each 64×64 = 4096 pixels; $X$ is $400 \times 4096$).

### **The Procedure**
1. Compute mean face $\vec{\mu}$ and center: $X_c = X - \vec{\mu}$.
2. Run PCA → top-$k$ eigenvectors (eigenfaces): $\vec{w}_1, ..., \vec{w}_k \in \mathbb{R}^{4096}$.
3. Represent each face by its $k$ PC scores: $\vec{z} = X_c W_k \in \mathbb{R}^k$.
4. Reconstruct: $\hat{\vec{x}} = \vec{z} W_k^T + \vec{\mu}$.

### **Compression Ratio**
- Original storage per face: $4096$ values.
- Compressed storage per face: $k$ values (PC scores) + shared eigenfaces (one-time cost).
- At $k=50$: $98.8\%$ fewer values per face, yet recognizable reconstruction.

### **Why Faces Work So Well**
Faces are structured — they all have eyes, noses, mouths in roughly the same locations. This correlational structure produces a rapidly decaying eigenvalue spectrum: the first few PCs capture most variance. Most random image patches would need far more PCs.

---

## 5. PCA as Preprocessing for Machine Learning

### **When PCA Preprocessing Helps**
| Scenario | Why PCA Helps |
| :--- | :--- |
| Features highly correlated ($\rho > 0.9$) | Removes redundancy; reduces overfitting in linear models |
| Downstream model assumes independence | Logistic Regression, Naive Bayes work better on uncorrelated features |
| $d \gg N$ (high-dim, small-$N$) | Reduces to $k < N$ components; avoids overfitting |
| Feature matrix too large for model | Compress to manageable size |
| Curse of dimensionality in distance-based models | KNN, kernel SVM work better in lower dims |

### **The Correct Pipeline**
```
Train set:  StandardScaler → PCA (fit + transform) → Model (fit)
Test set:   StandardScaler (transform only) → PCA (transform only) → Model (predict)
```
**Critical rule:** Fit the scaler and PCA on training data **only**. Apply the same fitted scaler and PCA to test data. Never re-fit on test data — that causes data leakage.

In sklearn: use `Pipeline([('scale', StandardScaler()), ('pca', PCA(k)), ('model', ...)])`.

### **When PCA Preprocessing Hurts**
- **Tree-based models (RF, XGBoost):** Trees are invariant to linear transformations; PCA rot doesn't help and loses interpretability.
- **Features already independent:** If $\Sigma$ is diagonal, PCA does nothing useful.
- **Important variance in low-eigenvalue direction:** If the class-discriminative signal lies in a low-variance direction (e.g., a subtle but informative feature), PCA will discard it.
- **Need interpretability:** PC axes are uninterpretable linear combinations — use feature selection instead.

---

## 6. Whitening (Sphering)

### **What Is Whitening?**
Whitening takes PCA one step further — it not only decorrelates the features but also normalizes each PC to have **unit variance**.

After standard PCA: covariance of $Z$ = $\Lambda$ (diagonal, but variances differ per component).

After whitening: covariance of $Z_{\text{white}}$ = $I$ (identity — unit variance in every direction).

### **Formula**
$$Z_{\text{white}} = Z \Lambda^{-1/2} = X W_k \Lambda_k^{-1/2}$$

where $\Lambda^{-1/2} = \text{diag}(\lambda_1^{-1/2}, ..., \lambda_k^{-1/2})$.

Equivalently, in terms of the original data:
$$Z_{\text{white}} = X \underbrace{W_k \Lambda_k^{-1/2}}_{\text{whitening matrix}}$$

### **Why Whitening?**
- **Gradient descent converges faster** when features have equal variance (avoids elongated loss surfaces).
- **Required for some algorithms:** ICA (Independent Component Analysis) assumes whitened input.
- **Feature equalizes scale within PCA space** — each PC contributes equally regardless of its variance.

### **Caution**
Whitening divides by $\sqrt{\lambda_i}$. For near-zero eigenvalues (noise directions), $\lambda_i \approx 0$ → division blows up. Always discard small-eigenvalue PCs before whitening. Use $k$ PCs that exceed the noise floor.

---

## 7. When PCA Hurts

### **Scenario 1: Variance ≠ Relevance**
Imagine a medical dataset with features [body weight, blood pressure, rare biomarker]. Weight has huge variance; the biomarker has tiny variance but is the actual disease predictor. PCA might discard the biomarker because it has low variance, causing a dramatic drop in prediction accuracy.

**Fix:** Use supervised methods (LDA, or feature importance from a model) to identify which features actually matter.

### **Scenario 2: Categorical Features**
PCA assumes continuous numerical data. Applying PCA to one-hot encoded data or ordinal features gives meaningless results (distances and variances are not meaningful for categorical data).

**Fix:** Use Multiple Correspondence Analysis (MCA) for categorical data.

### **Scenario 3: Non-Linear Structure**
Swiss roll, concentric circles, or any manifold: linear PCA cannot "unroll" the manifold. It will mix up points that are close on the manifold but far in Euclidean space.

**Fix:** Kernel PCA (Chapter 5 of PCA series if covered).

### **Scenario 4: Adversarial Outliers**
Even one outlier far from the data cloud can rotate the first PC completely toward that outlier, ruining the decomposition for the entire dataset.

**Fix:** Robust PCA (Chapter 4 of PCA series).

---

## 8. Interview Q&A

**Q1: What does PC1 represent?**
> PC1 is the direction in the original feature space along which the data has the largest variance. It is a unit vector whose components (loadings) tell you how much each original feature contributes to this direction.

**Q2: Why is the PCA biplot useful?**
> A biplot overlays the PC scores (data points in PC space) with loading vectors (arrows from origin for each original feature). The length and direction of each arrow shows how strongly and in what direction each original feature contributes to the shown PCs. It lets you interpret what the PC axes "mean" in terms of original features.

**Q3: Can PCA remove noise? How?**
> Yes. Signal lives in high-variance directions (large eigenvalues); noise spreads uniformly across all directions (small eigenvalues). By keeping only the top-$k$ PCs and reconstructing, we suppress the noise that lives in the truncated directions. The reconstruction error = variance in discarded directions = mostly noise variance.

**Q4: When is PCA as preprocessing harmful?**
> When the most class-discriminative signal lies in a low-variance direction (which PCA discards), when using tree-based models (which are rotation-invariant and don't benefit), or when interpretability is needed (PCs are uninterpretable mixtures of features).

**Q5: What is whitening and why is it used?**
> Whitening applies PCA then scales each PC to unit variance: $Z_{\text{white}} = XW_k \Lambda_k^{-1/2}$. The result has covariance $= I$. It's used to speed up gradient descent, remove scale differences between PCs, and as a preprocessing step for ICA.

**Q6: "95% explained variance" — does this mean 95% accuracy?**
> No. Explained variance ratio measures how much of the data's variance is retained, not prediction accuracy. A model trained on 95% EVR features may perform worse than 50% EVR features if the relevant signal lives in the low-variance directions.

---

In [ ]:
# ─── CELL 1: Biplot — Scores + Loadings in One Plot ───────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

iris = load_iris()
X_std = StandardScaler().fit_transform(iris.data)
pca = PCA(n_components=2)
Z = pca.fit_transform(X_std)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Left: Scatter of PC scores
colors = ['#e74c3c', '#2ecc71', '#3498db']
for cls, name, c in zip([0,1,2], iris.target_names, colors):
    mask = iris.target == cls
    axes[0].scatter(Z[mask, 0], Z[mask, 1], s=30, alpha=0.7, c=c, label=name)
evr = pca.explained_variance_ratio_
axes[0].set_xlabel(f'PC1 ({evr[0]*100:.1f}%)', fontsize=12)
axes[0].set_ylabel(f'PC2 ({evr[1]*100:.1f}%)', fontsize=12)
axes[0].set_title('PC Scores (Scatter)', fontsize=12)
axes[0].legend(); axes[0].grid(alpha=0.3)

# Right: Biplot — scores + loading arrows
for cls, name, c in zip([0,1,2], iris.target_names, colors):
    mask = iris.target == cls
    axes[1].scatter(Z[mask, 0], Z[mask, 1], s=20, alpha=0.4, c=c, label=name)

loadings = pca.components_.T  # d x k (4 x 2)
scale = 3.0  # Scale arrows for visibility
feature_names_short = ['Sepal L', 'Sepal W', 'Petal L', 'Petal W']
for j, fname in enumerate(feature_names_short):
    axes[1].annotate('', xy=(loadings[j,0]*scale, loadings[j,1]*scale),
                     xytext=(0., 0.),
                     arrowprops=dict(arrowstyle='->', color='darkred', lw=2))
    axes[1].text(loadings[j,0]*scale*1.1, loadings[j,1]*scale*1.1,
                 fname, fontsize=10, color='darkred', fontweight='bold')

axes[1].set_xlabel(f'PC1 ({evr[0]*100:.1f}%)', fontsize=12)
axes[1].set_ylabel(f'PC2 ({evr[1]*100:.1f}%)', fontsize=12)
axes[1].set_title('Biplot: Scores + Feature Loadings', fontsize=12)
axes[1].legend(loc='upper right', fontsize=9); axes[1].grid(alpha=0.3)
axes[1].set_xlim(-3.5, 3.5); axes[1].set_ylim(-3, 3)

plt.suptitle('Iris PCA Biplot\nArrows show how original features contribute to each PC', 
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("\nLoadings matrix (rows=PCs, cols=features):")
print(f"{'Feature':<15} {'PC1 loading':>12} {'PC2 loading':>12}")
for name, l1, l2 in zip(iris.feature_names, pca.components_[0], pca.components_[1]):
    print(f"{name:<15} {l1:>12.4f} {l2:>12.4f}")

In [ ]:
# ─── CELL 2: PCA for Noise Reduction — Signal Recovery ────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

np.random.seed(42)
N, d = 200, 50
k_true = 5   # true signal dimensionality

# Generate low-rank signal (true PCs) + noise
W_true = np.random.randn(d, k_true)  # true loading directions
Z_true = np.random.randn(N, k_true)  # true scores
X_signal = Z_true @ W_true.T         # clean signal (rank k_true)
X_noisy = X_signal + 1.5 * np.random.randn(N, d)  # add isotropic noise

# PCA reconstruction at different k
ks_try = [1, 3, 5, 10, 20]
errors_vs_signal = []
for k in range(1, d+1):
    pca = PCA(n_components=k).fit(X_noisy)
    X_rec = pca.inverse_transform(pca.transform(X_noisy))
    err = np.mean((X_rec - X_signal)**2)  # error vs TRUE signal (not noisy)
    errors_vs_signal.append(err)

best_k = np.argmin(errors_vs_signal) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Recovery error vs k
axes[0].plot(range(1, d+1), errors_vs_signal, lw=2, color='steelblue')
axes[0].axvline(k_true, color='green', ls='--', lw=2, label=f'True latent dim k={k_true}')
axes[0].axvline(best_k, color='red', ls=':', lw=2, label=f'Best recovery k={best_k}')
axes[0].set_xlabel('Number of PCs kept (k)')
axes[0].set_ylabel('MSE vs true signal (lower = better recovery)')
axes[0].set_title('PCA Denoising: Recovery Error vs k', fontsize=11)
axes[0].legend(); axes[0].grid(alpha=0.3)

# Right: Visual comparison for one sample
sample_idx = 0
x_original = X_signal[sample_idx]
x_noisy_sig = X_noisy[sample_idx]

pca_best = PCA(n_components=best_k).fit(X_noisy)
x_denoised = pca_best.inverse_transform(pca_best.transform(X_noisy))[sample_idx]

axes[1].plot(x_original, color='green', lw=2, label='True signal', alpha=0.8)
axes[1].plot(x_noisy_sig, color='gray', lw=1, alpha=0.5, label='Noisy observation')
axes[1].plot(x_denoised, color='red', lw=2, ls='--', label=f'PCA denoised (k={best_k})')
axes[1].set_xlabel('Feature index'); axes[1].set_ylabel('Value')
axes[1].set_title(f'Sample #{sample_idx}: Signal Recovery', fontsize=11)
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('PCA for Denoising: Optimal k recovers the true signal structure', 
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print(f"Best denoising k={best_k} (true signal k={k_true})")

In [ ]:
# ─── CELL 3: Eigenfaces — PCA for Face Compression ────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces
from sklearn.decomposition import PCA

# 400 face images, 64x64 pixels each
faces = fetch_olivetti_faces(shuffle=True, random_state=42)
X_faces = faces.data   # 400 x 4096
print(f"Faces dataset: {X_faces.shape} — {X_faces.shape[0]} faces, {X_faces.shape[1]} pixels each")

# Fit PCA models with varying k
ks = [1, 5, 20, 50, 100, 200]
pca_full = PCA().fit(X_faces)

fig, axes = plt.subplots(len(ks) + 1, 6, figsize=(15, 3*(len(ks)+1)))
face_indices = [0, 5, 10, 15, 20, 25]

# Row 0: original faces
for col, fi in enumerate(face_indices):
    axes[0, col].imshow(X_faces[fi].reshape(64, 64), cmap='gray')
    axes[0, col].set_title(f'Original {fi}' if col == 0 else f'Face {fi}', fontsize=8)
    axes[0, col].axis('off')
axes[0, 0].set_ylabel('Original', fontsize=10)

# Rows 1+: PCA reconstructions
for row, k in enumerate(ks):
    pca_k = PCA(n_components=k).fit(X_faces)
    X_recon = pca_k.inverse_transform(pca_k.transform(X_faces))
    evr = pca_k.explained_variance_ratio_.sum() * 100
    compression = X_faces.shape[1] / (k * (1 + X_faces.shape[0]/X_faces.shape[0]))
    
    for col, fi in enumerate(face_indices):
        axes[row+1, col].imshow(X_recon[fi].reshape(64, 64), cmap='gray', vmin=0, vmax=1)
        axes[row+1, col].axis('off')
    axes[row+1, 0].set_ylabel(f'k={k}\n{evr:.0f}% var', fontsize=9)

plt.suptitle('Eigenfaces: PCA Reconstruction at Increasing k\n(More PCs = Better reconstruction, less compression)',
             fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# Plot first eigenfaces
pca_50 = PCA(n_components=50).fit(X_faces)
fig2, axes2 = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes2.flatten()):
    eigenface = pca_50.components_[i].reshape(64, 64)
    ax.imshow(eigenface, cmap='RdBu_r')
    ax.set_title(f'Eigenface {i+1}\nλ={pca_50.explained_variance_[i]:.0f}', fontsize=8)
    ax.axis('off')
plt.suptitle('First 10 Eigenfaces (Principal Components of Face Space)', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
# ─── CELL 4: Whitening — Before vs After ──────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = load_iris().data  # 150 x 4
X_std = StandardScaler().fit_transform(X)

# Standard PCA
pca = PCA(n_components=2).fit(X_std)
Z = pca.transform(X_std)  # PC scores

# Whitening
Lambda_inv_sqrt = np.diag(1.0 / np.sqrt(pca.explained_variance_))
Z_white = Z @ Lambda_inv_sqrt

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Original standardized data
axes[0].scatter(X_std[:,0], X_std[:,1], s=15, alpha=0.5, c=load_iris().target, cmap='Set1')
axes[0].set_aspect('equal'); axes[0].grid(alpha=0.3)
axes[0].set_title('Original (Standardized)\nFeatures still correlated', fontsize=11)
cov0 = np.cov(X_std[:,:2].T)
axes[0].text(-2, 2.5, f'Cov[0,1]={cov0[0,1]:.3f}', fontsize=9, color='red')

# After PCA (decorrelated, but different variances)
axes[1].scatter(Z[:,0], Z[:,1], s=15, alpha=0.5, c=load_iris().target, cmap='Set1')
axes[1].set_aspect('equal'); axes[1].grid(alpha=0.3)
axes[1].set_title('After PCA\nDecorrelated, but PC1 >> PC2 variance', fontsize=11)
cov_z = np.cov(Z.T)
axes[1].text(-3.5, 1.5, f'Var(PC1)={cov_z[0,0]:.2f}', fontsize=9, color='red')
axes[1].text(-3.5, 1.1, f'Var(PC2)={cov_z[1,1]:.2f}', fontsize=9, color='blue')

# After whitening (decorrelated + unit variance)
axes[2].scatter(Z_white[:,0], Z_white[:,1], s=15, alpha=0.5, c=load_iris().target, cmap='Set1')
axes[2].set_aspect('equal'); axes[2].grid(alpha=0.3)
axes[2].set_title('After Whitening\nDecorrelated + unit variance in all directions', fontsize=11)
cov_w = np.cov(Z_white.T)
axes[2].text(-2, 2.5, f'Var(dim1)={cov_w[0,0]:.3f}', fontsize=9, color='red')
axes[2].text(-2, 2.1, f'Var(dim2)={cov_w[1,1]:.3f}', fontsize=9, color='blue')
axes[2].text(-2, 1.7, f'Cov={cov_w[0,1]:.3f}', fontsize=9, color='green')

plt.suptitle('Effect of PCA and Whitening on Data Distribution', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print("Covariance of whitened data (should be ≈ Identity):")
print(np.round(np.cov(Z_white.T), 4))